# Báo cáo Quy trình Tiền Xử lý Dữ liệu
## `prepare_articles.py` — Chuẩn bị dữ liệu cho PhoBERT

> **Ngày:** 09/03/2026  
> **Mục tiêu:** Lọc, làm sạch và chuẩn hóa `Dataset_articles.csv` để phục vụ huấn luyện mô hình phân loại văn bản tiếng Việt dựa trên PhoBERT (`train_v4.py`)

---

## Tổng quan quy trình

```
Dataset_articles.csv
        │
        ▼
  [1] Đọc dữ liệu thô
        │
        ▼
  [2] Chuẩn hoá tên cột
        │
        ▼
  [3] Ánh xạ Category → label
      (loại bỏ nhãn nhiễu)
        │
        ▼
  [4] Làm sạch text
      (HTML, URL, unicode, whitespace)
        │
        ▼
  [5] Lọc bài quá ngắn (< 100 ký tự)
        │
        ▼
  [6] Loại bỏ bài trùng lặp
        │
        ▼
  [7] Word Segmentation (underthesea)
        │
        ▼
 Dataset_articles_clean_YYYYMMDD_HHMM.csv
```

---

## Bước 1 — Đọc dữ liệu thô

### Cách làm
Script thử lần lượt các **encoding** khác nhau (`utf-8`, `utf-8-sig`, `cp1258`, `latin-1`) cho đến khi đọc thành công:

```python
for enc in ["utf-8", "utf-8-sig", "cp1258", "latin-1"]:
    try:
        df = pd.read_csv(SOURCE, encoding=enc, low_memory=False)
        break
    except UnicodeDecodeError:
        continue
```

### Tại sao phải làm vậy?
Dữ liệu thu thập từ báo điện tử Việt Nam thường không đồng nhất về mã hoá ký tự. Tiếng Việt có nhiều dấu tổ hợp đặc biệt; nếu encode sai, toàn bộ cột `content` sẽ xuất hiện ký tự "◻", "ï¿½" hoặc raise `UnicodeDecodeError`. Cơ chế **fallback** đảm bảo file luôn được đọc thành công dù nguồn gốc CSV là Windows (CP1258) hay Unix (UTF-8).

### Lợi ích
| Lợi ích | Giải thích |
|---|---|
| Tính bền vững | Script không bị crash khi encoding thay đổi |
| Tính đúng đắn | Không bị mất / méo ký tự tiếng Việt |
| Khả năng tái sử dụng | Chạy được trên mọi file CSV tiếng Việt |

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

SOURCE = Path("data/data/Dataset_articles.csv")

# Thử lần lượt các encoding
df = None
used_enc = None
for enc in ["utf-8", "utf-8-sig", "cp1258", "latin-1"]:
    try:
        df = pd.read_csv(SOURCE, encoding=enc, low_memory=False)
        used_enc = enc
        break
    except (UnicodeDecodeError, FileNotFoundError):
        continue

if df is not None:
    print(f"Encoding thành công : {used_enc}")
    print(f"Số dòng x cột       : {df.shape}")
    print(f"Tên cột             : {list(df.columns)}")
    df.head(3)
else:
    print("⚠️  Không tìm thấy file nguồn — demo sẽ bỏ qua bước này.")

Encoding thành công : utf-8
Số dòng x cột       : (313320, 9)
Tên cột             : ['Unnamed: 0', 'URL', 'Title', 'Summary', 'Contents', 'Date', 'Author(s)', 'Category', 'Tags']


---

## Bước 2 — Chuẩn hoá tên cột

### Cách làm
Đổi tên cột từ dạng Pascal/CamelCase (tên gốc của CSV) sang dạng `snake_case` chuẩn của Python:

```python
df = df.rename(columns={
    "Title":      "title",
    "Summary":    "description",
    "Contents":   "content",
    "Category":   "category_raw",
    ...
})
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
```

### Tại sao phải làm vậy?
- Pandas tạo cột `Unnamed: 0` khi CSV được lưu kèm index không mong muốn. Cột này là rác và cần được xóa.
- Tên cột nhất quán (`snake_case`) giúp các bước sau (và `train_v4.py`) truy cập dễ dàng: `df["content"]` thay vì `df["Contents"]`.
- Cột `Category` được đổi thành `category_raw` để phân biệt rõ với cột `label` đã qua ánh xạ.

### Lợi ích
- Code dễ đọc, nhất quán
- Tránh lỗi `KeyError` khi tên cột có khoảng trắng hoặc hoa/thường không đồng đều
- `train_v4.py` không cần quan tâm đến tên cột gốc

In [2]:
# Demo: so sánh trước/sau khi đổi tên cột
sample_before = ["Title", "Summary", "Contents", "Category", "URL", "Date", "Author(s)", "Tags", "Unnamed: 0"]
rename_map = {
    "Title":      "title",
    "Summary":    "description",
    "Contents":   "content",
    "Category":   "category_raw",
    "URL":        "url",
    "Date":       "date",
    "Author(s)":  "author",
    "Tags":       "tags",
}

print(f"{'Tên cột GỐC':<20} {'Tên cột MỚI':<20} {'Ghi chú'}")
print("-" * 65)
for col in sample_before:
    if col == "Unnamed: 0":
        print(f"{col:<20} {'--- XÓA ---':<20} Index thừa từ to_csv(index=True)")
    else:
        new_name = rename_map.get(col, col)
        print(f"{col:<20} {new_name:<20}")

Tên cột GỐC          Tên cột MỚI          Ghi chú
-----------------------------------------------------------------
Title                title               
Summary              description         
Contents             content             
Category             category_raw        
URL                  url                 
Date                 date                
Author(s)            author              
Tags                 tags                
Unnamed: 0           --- XÓA ---          Index thừa từ to_csv(index=True)


---

## Bước 3 — Ánh xạ Category → Label & Loại bỏ nhãn nhiễu

### Cách làm
Dùng dictionary `CATEGORY_TO_LABEL` để gộp nhiều `category_raw` thành một **label ngữ nghĩa** thống nhất. Các category không có trong dictionary sẽ bị loại bỏ:

```python
CATEGORY_TO_LABEL = {
    "Kinh doanh":     "kinh_doanh",
    "Bất động sản":   "kinh_doanh",   # gộp vào kinh_doanh
    "Xã hội":      → LOẠI BỎ (không có trong dict)
    ...
}
df["label"] = df["category_raw"].map(CATEGORY_TO_LABEL)
df = df.dropna(subset=["label"])   # loại dòng không map được
```

### Tại sao loại bỏ "Xã hội" và "Bạn đọc"?
- **"Xã hội"** là category quá rộng, chứa bài từ nhiều chủ đề khác nhau (giáo dục, pháp luật, y tế…). Nhãn này **không đặc trưng** — mô hình sẽ không học được pattern rõ ràng.
- **"Bạn đọc"** là ý kiến độc giả, thường ngắn, viết tay, thiếu cấu trúc → nhiễu cao.
- **"Tản mạn - Chuyện dọc đường"** là dạng tản văn, mang tính cá nhân, không thuộc thể loại tin tức.

### Tại sao gộp category?
Các category như `"Bất động sản"`, `"Thông tin doanh nghiệp"`, `"Sổ tay kinh tế"` đều nói về **kinh doanh**. Gộp lại:
1. Tăng số mẫu mỗi nhãn → mô hình học tốt hơn
2. Giảm độ mất cân bằng nhãn (imbalance)
3. Nhãn có ý nghĩa nhất quán về mặt ngữ nghĩa

### Lợi ích tổng thể
| Vấn đề | Nếu không làm | Sau khi làm |
|---|---|---|
| Nhãn mơ hồ | Model nhầm lẫn, accuracy giả cao | Nhãn rõ ràng, model học pattern thật |
| Imbalance | Loss bị chiều bởi class đa số | Các class gần bằng nhau hơn |
| Nhãn nhiễu | Model overfit noise | Dữ liệu sạch, generalise tốt |

In [3]:
CATEGORY_TO_LABEL = {
    "Giáo dục":                 "giao_duc",
    "Kinh doanh":               "kinh_doanh",
    "Bất động sản":             "kinh_doanh",
    "Sổ tay kinh tế":           "kinh_doanh",
    "Thông tin doanh nghiệp":   "kinh_doanh",
    "Công đoàn":                "lao_dong",
    "Lao Động & Đời sống":      "lao_dong",
    "Lao Động Xuân":            "lao_dong",
    "Lao Động cuối tuần":       "lao_dong",
    "Tin tức việc làm":         "lao_dong",
    "Người Việt tử tế":         "nhan_ai_cong_dong",
    "Quỹ TLV":                  "nhan_ai_cong_dong",
    "Tấm Lòng Vàng":            "nhan_ai_cong_dong",
    "Pháp luật":                "phap_luat",
    "Phóng sự":                 "phap_luat",
    "Phóng sự - Điều tra":      "phap_luat",
    "Sức khỏe":                 "suc_khoe_gia_dinh",
    "Gia đình - Hôn nhân":      "suc_khoe_gia_dinh",
    "Thế giới":                 "the_gioi",
    "Thể thao":                 "the_thao",
    "Thời sự":                  "thoi_su",
    "Diễn đàn":                 "thoi_su",
    "Sự kiện Bình luận":        "thoi_su",
    "Tin địa phương":           "thoi_su",
    "Văn hóa - Giải trí":       "van_hoa_giai_tri",
    "Media":                    "van_hoa_giai_tri",
    "Du lịch":                  "van_hoa_giai_tri",
    "Công nghệ":                "xe_cong_nghe",
    "Xe +":                     "xe_cong_nghe",
}

# Hiển thị nhãn cuối và số category gộp vào từng nhãn
from collections import Counter
label_group = Counter(CATEGORY_TO_LABEL.values())

print(f"{'Nhãn (label)':<25} {'Số category gộp vào':<22} {'Danh sách category'}")
print("-" * 90)
for label in sorted(label_group):
    cats = [c for c, l in CATEGORY_TO_LABEL.items() if l == label]
    print(f"{label:<25} {label_group[label]:<22} {', '.join(cats)}")

print(f"\n→ Tổng: {len(CATEGORY_TO_LABEL)} category gốc → {len(set(CATEGORY_TO_LABEL.values()))} nhãn cuối")
print(f"  Các category bị loại: Xã hội, Bạn đọc, Tản mạn - Chuyện dọc đường")

Nhãn (label)              Số category gộp vào    Danh sách category
------------------------------------------------------------------------------------------
giao_duc                  1                      Giáo dục
kinh_doanh                4                      Kinh doanh, Bất động sản, Sổ tay kinh tế, Thông tin doanh nghiệp
lao_dong                  5                      Công đoàn, Lao Động & Đời sống, Lao Động Xuân, Lao Động cuối tuần, Tin tức việc làm
nhan_ai_cong_dong         3                      Người Việt tử tế, Quỹ TLV, Tấm Lòng Vàng
phap_luat                 3                      Pháp luật, Phóng sự, Phóng sự - Điều tra
suc_khoe_gia_dinh         2                      Sức khỏe, Gia đình - Hôn nhân
the_gioi                  1                      Thế giới
the_thao                  1                      Thể thao
thoi_su                   4                      Thời sự, Diễn đàn, Sự kiện Bình luận, Tin địa phương
van_hoa_giai_tri          3                      Văn hóa - 

---

## Bước 4 — Làm sạch văn bản (`clean_text`)

Đây là bước **quan trọng nhất** và phức tạp nhất. Hàm `clean_text()` áp dụng **12 lớp xử lý** theo thứ tự từ cấp độ encoding xuống cấp độ khoảng trắng.

```
Văn bản thô
    │
    ├─ [4.1] Chuẩn hoá Unicode NFC
    ├─ [4.2] Xử lý HTML entities
    ├─ [4.3] Xoá HTML tags
    ├─ [4.4] Xoá URL & email
    ├─ [4.5] Xử lý unicode space đặc biệt
    ├─ [4.6] Xoá zero-width characters
    ├─ [4.7] Chuẩn hoá dấu ngoặc và gạch ngang
    ├─ [4.8] Xoá control characters
    ├─ [4.9] Xoá combining marks thừa
    ├─ [4.10] Rút gọn dấu câu lặp
    ├─ [4.11] Chuẩn hoá xuống dòng
    └─ [4.12] Xoá khoảng trắng thừa
    │
Văn bản sạch
```

### Ví dụ: Dữ liệu thô trước khi làm sạch

Dưới đây là một ví dụ bài báo **thực tế** khi mới crawl — chứa đầy đủ các loại nhiễu mà hàm `clean_text` sẽ xử lý:


In [4]:
raw_example = (
    "<p><strong>Hà\u200bNội</strong>&nbsp;\u2014 Theo "
    "<a href='https://vnexpress.net/kinh-te/bat-dong-san.html'>nguồn tin từ VnExpress</a>,\r\n"
    "gi\u00e1 c\u0103n hộ tại Hà Nội trong năm 2024\u20132025 tăng mạnh!!!!!! "
    "Nhiều chuyên gia nhận định \u201cthị trường vẫn còn tiềm năng\u201d.\n\n"
    "Liên hệ tòa soạn: toasoan@baovietnam.vn ho\u1eb7c gh\u00e9 th\u0103m www.baovietnam.vn/lien-he\n"
    "&#8211; Xem thêm: &lt;bảng giá chi tiết&gt; ở link trên&#160;\n"
    "Từ gh\u00e9p cần tách: kinh tế, học sinh, ô tô, bộ trưởng\t</p>"
)

print("=" * 70)
print("DỮ LIỆU THÔ (trước khi làm sạch):")
print("=" * 70)
print(raw_example)
print()
print(f"Độ dài: {len(raw_example)} ký tự")
print()
print("Các vấn đề hiện có:")
problems = [
    "- HTML tags: <p>, <strong>, <a href=...>",
    "- HTML entities: &nbsp;, &lt;, &gt;, &#8211;, &#160;",
    "- Zero-width space: \\u200b trong 'Hà\\u200bNội'",
    "- Windows line ending: \\r\\n",
    "- Double newline (paragraph break): \\n\\n",
    "- Typography dash: \\u2014 (—), en-dash \\u2013 (–)",
    "- Typography quotes: \\u201c, \\u201d (\" \")",
    "- Repeated punctuation: !!!!!!",
    "- URL: https://vnexpress.net/...",
    "- Email: toasoan@baovietnam.vn",
    "- Tab character: \\t",
    "- Non-breaking space: \\xa0 / &#160;",
]
for p in problems:
    print(p)


DỮ LIỆU THÔ (trước khi làm sạch):
<p><strong>Hà​Nội</strong>&nbsp;— Theo <a href='https://vnexpress.net/kinh-te/bat-dong-san.html'>nguồn tin từ VnExpress</a>,
giá căn hộ tại Hà Nội trong năm 2024–2025 tăng mạnh!!!!!! Nhiều chuyên gia nhận định “thị trường vẫn còn tiềm năng”.

Liên hệ tòa soạn: toasoan@baovietnam.vn hoặc ghé thăm www.baovietnam.vn/lien-he
&#8211; Xem thêm: &lt;bảng giá chi tiết&gt; ở link trên&#160;
Từ ghép cần tách: kinh tế, học sinh, ô tô, bộ trưởng	</p>

Độ dài: 443 ký tự

Các vấn đề hiện có:
- HTML tags: <p>, <strong>, <a href=...>
- HTML entities: &nbsp;, &lt;, &gt;, &#8211;, &#160;
- Zero-width space: \u200b trong 'Hà\u200bNội'
- Windows line ending: \r\n
- Double newline (paragraph break): \n\n
- Typography dash: \u2014 (—), en-dash \u2013 (–)
- Typography quotes: \u201c, \u201d (" ")
- Repeated punctuation: !!!!!!
- URL: https://vnexpress.net/...
- Email: toasoan@baovietnam.vn
- Tab character: \t
- Non-breaking space: \xa0 / &#160;


### 4.1 — Chuẩn hoá Unicode NFC

```python
text = unicodedata.normalize("NFC", text)
```

**Vấn đề:** Tiếng Việt có thể biểu diễn cùng 1 ký tự theo 2 cách:
- **NFC**: `ộ` = 1 code point `U+1ED9` (precomposed)
- **NFD**: `ộ` = 3 code points: `o` + `̣` (combining dot below) + `̂` (combining circumflex)

Nếu không chuẩn hoá, hai từ trông giống hệt nhau có thể **không bằng nhau** khi so sánh chuỗi, gây lỗi tokenizer và duplicate không bị phát hiện.

**Lợi ích:** Tất cả ký tự tiếng Việt về cùng dạng chuẩn → tokenizer PhoBERT nhận diện đúng, bước dedup hoạt động chính xác.

**Ví dụ dữ liệu trước/sau bước 4.1:**


In [ ]:
import unicodedata

# Ví dụ thực tế: cùng chữ "Việt Nam" nhưng 2 cách encode khác nhau
nfd_viet = "Vie\u0302\u0323t Nam"   # 'e' + combining circumflex + combining dot below
nfc_viet = "Vi\u1ec7t Nam"          # ệ precomposed

print("══ Ví dụ: chữ 'Việt Nam' dạng NFD vs NFC ══\n")
print(f"  NFD (tổ hợp)  : '{nfd_viet}'  — {len(nfd_viet)} ký tự — code points: {[hex(ord(c)) for c in nfd_viet]}")
print(f"  NFC (chuẩn)   : '{nfc_viet}'  — {len(nfc_viet)} ký tự — code points: {[hex(ord(c)) for c in nfc_viet]}")
print(f"  Trông giống nhau? Có")
print(f"  So sánh bằng == ? {nfd_viet == nfc_viet}  ← đây là vấn đề!")
print()

# Ví dụ ảnh hưởng thực tế: dedup không bắt được
sentence_nfd = "Kinh te\u0301 Vie\u0302\u0323t Nam ta\u0306ng tru\u01b0\u0309ng"
sentence_nfc = "Kinh tế Việt Nam tăng trưởng"
print(f"── Câu NFD: '{sentence_nfd}'")
print(f"── Câu NFC: '{sentence_nfc}'")
print(f"   Chúng có bằng nhau? {sentence_nfd == sentence_nfc}  ← dedup sẽ bỏ sót!\n")

# Sau khi normalize
fixed = unicodedata.normalize("NFC", sentence_nfd)
print(f"── Sau normalize('NFC'): '{fixed}'")
print(f"   Bây giờ bằng nhau?   {fixed == sentence_nfc}  ✓")


In [5]:
import unicodedata

# Minh họa NFC vs NFD
nfd_char = "\u006f\u0323\u0302"   # 'o' + combining dot below + combining circumflex
nfc_char = "\u1ed9"               # ộ precomposed

print(f"NFD  : '{nfd_char}'  — len={len(nfd_char)} — code points: {[hex(ord(c)) for c in nfd_char]}")
print(f"NFC  : '{nfc_char}'  — len={len(nfc_char)} — code points: {[hex(ord(c)) for c in nfc_char]}")
print(f"Trông giống nhau? {'Có' if nfd_char == nfc_char else 'Không'}")
print(f"Sau normalize NFC bằng nhau? {unicodedata.normalize('NFC', nfd_char) == nfc_char}")

NFD  : 'ộ'  — len=3 — code points: ['0x6f', '0x323', '0x302']
NFC  : 'ộ'  — len=1 — code points: ['0x1ed9']
Trông giống nhau? Không
Sau normalize NFC bằng nhau? True


### 4.2 — Xử lý HTML Entities

```python
text = re.sub(r"&nbsp;|&#160;", " ", text)
text = re.sub(r"&amp;",         "&", text)
text = re.sub(r"&lt;",          "<", text)
text = re.sub(r"&gt;",          ">", text)
text = re.sub(r"&quot;|&#34;",  '"', text)
text = re.sub(r"&#\d+;|&[a-zA-Z]+;", " ", text)   # các entity còn lại
```

**Vấn đề:** Dữ liệu crawl từ website HTML thường chứa entity codes như `&nbsp;`, `&amp;`, `&#8211;`. PhoBERT **không hiểu** `&amp;` là `&` — nó sẽ tokenize thành các token lạ, làm giảm chất lượng embedding.

**Lợi ích:** Văn bản trả về dạng đọc được cho người, phù hợp với training corpus của PhoBERT (vốn là văn bản sạch từ Wikipedia/news).

**Ví dụ dữ liệu trước/sau bước 4.2:**


In [ ]:
import re

# Ví dụ bài báo chứa HTML entities thực tế
before_entities = (
    "Giá nhà tăng&nbsp;mạnh, nhiều người lo ngại &lt;bong bóng&gt; địa ốc.\n"
    "Chỉ số VN&#8209;Index đạt 1.200&#160;điểm.\n"
    "Tăng trưởng GDP đạt 6,5&#37; &#8211; mức cao nhất trong 5 năm.\n"
    "Công ty &quot;ABC Holdings&quot; thông báo &amp; tuyển dụng nhân sự."
)

def clean_html_entities(text):
    text = re.sub(r"&nbsp;|&#160;", " ", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)
    text = re.sub(r'&quot;|&#34;', '"', text)
    text = re.sub(r"&#\d+;|&[a-zA-Z]+;", " ", text)
    return text

after_entities = clean_html_entities(before_entities)

print("TRƯỚC (dữ liệu thô):")
print("-" * 60)
print(before_entities)
print()
print("SAU (đã xử lý HTML entities):")
print("-" * 60)
print(after_entities)
print()
print("Nhận xét:")
print("  &nbsp; → dấu cách bình thường")
print("  &lt;&gt; → < > (dấu ngoặc nhọn thật)")
print("  &#8211; → (dấu gạch ngang thật, sẽ xử lý tiếp ở 4.6)")
print("  &amp; → & (dấu và thật)")
print("  &quot; → \" (dấu nháy kép thật)")


In [6]:
import re

dirty = "Giá nhà tăng&nbsp;mạnh&amp;, nhiều người lo ngại &lt;bong bóng&gt; địa ốc &#8211; theo báo cáo Q4&#160;2025."

def demo_html_entities(text: str) -> str:
    text = re.sub(r"&nbsp;|&#160;", " ", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)
    text = re.sub(r"&quot;|&#34;", '"', text)
    text = re.sub(r"&#\d+;|&[a-zA-Z]+;", " ", text)
    return text

print("TRƯỚC:", dirty)
print("SAU  :", demo_html_entities(dirty))

TRƯỚC: Giá nhà tăng&nbsp;mạnh&amp;, nhiều người lo ngại &lt;bong bóng&gt; địa ốc &#8211; theo báo cáo Q4&#160;2025.
SAU  : Giá nhà tăng mạnh&, nhiều người lo ngại <bong bóng> địa ốc   theo báo cáo Q4 2025.


### 4.3 — Xoá HTML Tags

```python
text = re.sub(r"<[^>]+>", " ", text)
```

**Vấn đề:** Nội dung crawl thường chứa các thẻ HTML như `<p>`, `<br>`, `<strong>`, `<a href="...">`. Đây là markup không mang thông tin ngữ nghĩa cho mô hình.

**Lợi ích:** Loại bỏ toàn bộ cú pháp HTML, chỉ giữ lại nội dung văn bản thuần túy.

**Ví dụ dữ liệu trước/sau bước 4.3:**


In [ ]:
import re

before_html = (
    '<div class="article-body">'
    '<h2>Thị trường bất động sản 2025</h2>'
    '<p>Theo <strong>chuyên gia</strong> kinh tế <em>Nguyễn Văn A</em>, '
    'thị trường đang <span style="color:red">bứt phá mạnh</span>.<br/>'
    'Giá căn hộ tại <a href="https://example.com" target="_blank">Hà Nội</a> '
    'tăng trung bình <b>15%</b> so với năm trước.</p>'
    '<ul><li>Khu vực trung tâm: +20%</li><li>Ngoại ô: +10%</li></ul>'
    '</div>'
)

after_html = re.sub(r"<[^>]+>", " ", before_html)
after_html = re.sub(r" {2,}", " ", after_html).strip()

print("TRƯỚC (chứa HTML tags):")
print("-" * 60)
print(before_html)
print()
print("SAU (đã xóa HTML tags):")
print("-" * 60)
print(after_html)
print()
print(f"Giảm từ {len(before_html)} → {len(after_html)} ký tự "
      f"({len(before_html)-len(after_html)} ký tự markup đã bị loại)")


In [7]:
dirty_html = "<p><strong>Hà Nội</strong> — Theo <a href='https://example.com'>nguồn tin</a>, <br/>giá xăng đã tăng 500 đồng/lít.</p>"
clean_html = re.sub(r"<[^>]+>", " ", dirty_html)
clean_html = re.sub(r" {2,}", " ", clean_html).strip()

print("TRƯỚC:", dirty_html)
print("SAU  :", clean_html)

TRƯỚC: <p><strong>Hà Nội</strong> — Theo <a href='https://example.com'>nguồn tin</a>, <br/>giá xăng đã tăng 500 đồng/lít.</p>
SAU  : Hà Nội — Theo nguồn tin , giá xăng đã tăng 500 đồng/lít.


### 4.4 — Xoá URL và Email

```python
text = re.sub(r"https?://\S+|www\.\S+", " ", text)
text = re.sub(r"\S+@\S+\.\S+",         " ", text)
```

**Vấn đề:** URL và địa chỉ email trong bài báo:
- Chiếm nhiều token nhưng không mang nội dung phân loại chủ đề
- Tạo ra các **rare tokens** (domain, path) làm phong phú từ điển một cách vô ích
- PhoBERT tokenizer sẽ phân mảnh URL thành nhiều subword tokens → lãng phí sequence length (tối đa 256 token)

**Lợi ích:** Tiết kiệm sequence length cho nội dung thật sự; giảm noise.

**Ví dụ dữ liệu trước/sau bước 4.4:**


In [ ]:
import re

before_url = (
    "Thị trường chứng khoán hồi phục mạnh theo tin tại https://cafef.vn/thi-truong/phuc-hoi-2025.html. "
    "Chỉ số VN-Index tăng 1.2% trong phiên sáng. "
    "Xem thêm tại www.laodong.vn/kinh-te/chung-khoan hoặc gửi ý kiến về toasoan@laodong.vn. "
    "Báo cáo đầy đủ: https://ssc.gov.vn/ubck/faces/oracle/webcenter/portalapp/pages/download-report?id=ABC123"
)

def remove_urls_emails(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

after_url = remove_urls_emails(before_url)

print("TRƯỚC (chứa URL và email):")
print("-" * 70)
print(before_url)
print()
print("SAU (đã xóa URL và email):")
print("-" * 70)
print(after_url)
print()
print("Nhận xét:")
print("  - Nội dung chủ đề 'chứng khoán tăng' vẫn được bảo toàn")
print("  - Đường dẫn dài, path URL, query string đã bị loại hoàn toàn")
print("  - Địa chỉ email đã bị xóa → không tạo token lạ cho PhoBERT")


In [8]:
texts_with_url = [
    "Đọc thêm tại https://vnexpress.net/kinh-te/bat-dong-san-2025.html",
    "Liên hệ: toa-soan@laodong.vn hoặc www.laodong.vn/lien-he",
]

for t in texts_with_url:
    cleaned = re.sub(r"https?://\S+|www\.\S+", "[URL]", t)
    cleaned = re.sub(r"\S+@\S+\.\S+", "[EMAIL]", cleaned)
    print(f"TRƯỚC: {t}")
    print(f"SAU  : {cleaned}")
    print()

TRƯỚC: Đọc thêm tại https://vnexpress.net/kinh-te/bat-dong-san-2025.html
SAU  : Đọc thêm tại [URL]

TRƯỚC: Liên hệ: toa-soan@laodong.vn hoặc www.laodong.vn/lien-he
SAU  : Liên hệ: [EMAIL] hoặc [URL]



### 4.5 — Xử lý Unicode whitespace đặc biệt & Zero-width characters

```python
# Non-breaking space và unicode space đặc biệt → dấu cách thường
text = text.replace("\xa0", " ")
text = re.sub(r"[\u2000-\u200b\u202f\u205f\u3000\ufeff]", " ", text)

# Zero-width chars (vô hình, không có chiều rộng) → xóa hoàn toàn
text = re.sub(r"[\u200c-\u200f\u2028\u2029]", "", text)
```

**Vấn đề:** Các ký tự này có thể xuất hiện do copy-paste từ Word, PDF, hay các CMS báo điện tử. Chúng:
- **Vô hình** với mắt người nhưng **có thật** trong string → gây lỗi tokenizer
- Zero-width joiner (`\u200d`) có thể làm 2 từ dính vào nhau trong khi trông như có khoảng cách
- `\ufeff` (BOM - Byte Order Mark) ở đầu file sẽ làm hỏng token đầu tiên

**Lợi ích:** Loại bỏ các "bẫy vô hình" không ảnh hưởng nội dung nhưng phá vỡ xử lý chuỗi.

**Ví dụ dữ liệu trước/sau bước 4.5:**


In [9]:
# Minh họa zero-width character
invisible_trap = "kinh\u200bdoanh"   # 'kinh' + zero-width space + 'doanh'
normal = "kinhdoanh"

print(f"Chuỗi 'bẫy' trông như: '{invisible_trap}'")
print(f"Chuỗi thường          : '{normal}'")
print(f"Chúng có bằng nhau không? {invisible_trap == normal}")
print(f"Độ dài 'bẫy': {len(invisible_trap)} | Độ dài thường: {len(normal)}")

# Sau khi xóa zero-width
cleaned = re.sub(r"[\u200c-\u200f\u2028\u2029\u200b]", "", invisible_trap)
print(f"\nSau khi làm sạch: '{cleaned}' == '{normal}'? {cleaned == normal}")

Chuỗi 'bẫy' trông như: 'kinh​doanh'
Chuỗi thường          : 'kinhdoanh'
Chúng có bằng nhau không? False
Độ dài 'bẫy': 10 | Độ dài thường: 9

Sau khi làm sạch: 'kinhdoanh' == 'kinhdoanh'? True


### 4.6 — Chuẩn hoá dấu câu và ký tự đặc biệt

```python
# Dấu ngoặc typography → ASCII
text = text.replace("\u201c", '"').replace("\u201d", '"')  # " " → "
text = text.replace("\u2018", "'").replace("\u2019", "'")  # ' ' → '
text = text.replace("\u00ab", '"').replace("\u00bb", '"')  # « » → "

# Dấu gạch ngang dài → dấu gạch ngang ASCII
text = text.replace("\u2013", "-").replace("\u2014", "-")  # – — → -
text = text.replace("\u2026", "...")                        # … → ...

# Dấu câu lặp quá nhiều → rút gọn
text = re.sub(r"([!?.…]){3,}", r"\1\1", text)
```

**Tại sao?** PhoBERT được huấn luyện trên corpus dùng ASCII punctuation chuẩn. Khi gặp dấu ngoặc kiểu typography như `"`, `"`, `—`, `…`, tokenizer có thể tạo ra **UNK token** hoặc subword lạ, làm giảm chất lượng embedding. Chuẩn hoá về ASCII đảm bảo dấu câu nhất quán với pre-training corpus.

In [10]:
import re

DQUOTE = chr(34)   # ASCII double-quote: "
SQUOTE = chr(39)   # ASCII single-quote: '

typography_samples = [
    ("\u201cN\u1ec1n kinh t\u1ebf\u201d t\u0103ng tr\u01b0\u1edfng",   "Typography quotes \u2192 ASCII quotes"),
    ("2024\u20132025",                                                    "En-dash \u2192 hyphen"),
    ("Th\u1eadt tuy\u1ec7t v\u1eddi!!!!!!",                             "D\u1ea5u c\u00e2u l\u1eb7p \u2192 r\u00fat g\u1ecdn"),
    ("\u2026 theo Reuters",                                               "Ellipsis unicode \u2192 ..."),
]

for original, desc in typography_samples:
    text = original
    text = text.replace("\u201c", DQUOTE).replace("\u201d", DQUOTE)
    text = text.replace("\u2018", SQUOTE).replace("\u2019", SQUOTE)
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u2026", "...")
    text = re.sub(r"([!?....]){3,}", r"\1\1", text)
    print(f"[{desc}]")
    print(f"  TR\u01af\u1edaC: {original!r}")
    print(f"  SAU  : {text!r}")
    print()

[Typography quotes → ASCII quotes]
  TRƯỚC: '“Nền kinh tế” tăng trưởng'
  SAU  : '"Nền kinh tế" tăng trưởng'

[En-dash → hyphen]
  TRƯỚC: '2024–2025'
  SAU  : '2024-2025'

[Dấu câu lặp → rút gọn]
  TRƯỚC: 'Thật tuyệt vời!!!!!!'
  SAU  : 'Thật tuyệt vời!!'

[Ellipsis unicode → ...]
  TRƯỚC: '… theo Reuters'
  SAU  : '.. theo Reuters'



### 4.7 — Xoá Control Characters

```python
text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
```

**Vấn đề:** Ký tự điều khiển (`\t` tab, `\r` carriage return, `\x01` SOH, v.v.) xuất hiện trong dữ liệu crawl hoặc copy từ PDF. Chúng không mang ý nghĩa ngữ nghĩa và có thể khiến CSV bị parse sai (đặc biệt `\r\n`).

### 4.8 — Chuẩn hoá xuống dòng và khoảng trắng

```python
text = re.sub(r"\r\n|\r", "\n", text)   # chuẩn hoá line ending
text = re.sub(r"\n{2,}", ". ", text)    # đoạn văn mới → dấu chấm
text = re.sub(r"\n", " ", text)         # xuống dòng đơn → dấu cách
text = re.sub(r" {2,}", " ", text)      # nhiều cách → 1 cách
```

**Tại sao biến `\n\n` thành `. ` thay vì xóa?** PhoBERT hoạt động ở cấp độ **câu**. Ranh giới đoạn văn là dấu hiệu kết thúc câu quan trọng. Chuyển thành `. ` giúp bảo toàn thông tin cấu trúc văn bản mà không làm tăng độ phức tạp.

In [11]:
import re

multiline_text = (
    "Th\u1ecb tr\u01b0\u1eddng b\u1ea5t \u0111\u1ed9ng s\u1ea3n 2025\r\n"
    "Gi\u00e1 c\u0103n h\u1ed9 t\u1ea1i H\u00e0 N\u1ed9i  t\u0103ng m\u1ea1nh\n\n"
    "Nguy\u00ean nh\u00e2n ch\u00ednh:\n- C\u1ea7u cao\n- Cung th\u1ea5p\n\n"
    "Chuy\u00ean gia nh\u1eadn \u0111\u1ecbnh..."
)

def clean_whitespace(text):
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"\n{2,}", ". ", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

print("TR\u01af\u1edaC:")
print(repr(multiline_text))
print("\nSAU:")
print(repr(clean_whitespace(multiline_text)))
print("\nHi\u1ec3n th\u1ecb:")
print(clean_whitespace(multiline_text))

TRƯỚC:
'Thị trường bất động sản 2025\r\nGiá căn hộ tại Hà Nội  tăng mạnh\n\nNguyên nhân chính:\n- Cầu cao\n- Cung thấp\n\nChuyên gia nhận định...'

SAU:
'Thị trường bất động sản 2025 Giá căn hộ tại Hà Nội tăng mạnh. Nguyên nhân chính: - Cầu cao - Cung thấp. Chuyên gia nhận định...'

Hiển thị:
Thị trường bất động sản 2025 Giá căn hộ tại Hà Nội tăng mạnh. Nguyên nhân chính: - Cầu cao - Cung thấp. Chuyên gia nhận định...


### Tổng hợp hàm `clean_text` đầy đủ

In [12]:
import re, unicodedata

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"&nbsp;|&#160;", " ", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)
    text = re.sub(r"&quot;|&#34;", chr(34), text)
    text = re.sub(r"&#\d+;|&[a-zA-Z]+;", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"[\u2000-\u200b\u202f\u205f\u3000\ufeff]", " ", text)
    text = re.sub(r"[\u200c-\u200f\u2028\u2029]", "", text)
    text = text.replace("\u201c", chr(34)).replace("\u201d", chr(34))
    text = text.replace("\u2018", chr(39)).replace("\u2019", chr(39))
    text = text.replace("\u00ab", chr(34)).replace("\u00bb", chr(34))
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u2026", "...")
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
    text = re.sub(r"[\u0300-\u036f]", "", text)
    text = re.sub(r"([!?....]){3,}", r"\1\1", text)
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"\n{2,}", ". ", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

# Test t\u1ed5ng h\u1ee3p
sample = (
    "<p><strong>H\u00e0\u200bN\u1ed9i</strong>&nbsp;\u2014 Theo "
    "<a href='https://vnexpress.net'>ngu\u1ed3n tin</a>,\n"
    "gi\u00e1 c\u0103n h\u1ed9 n\u0103m 2024\u20132025 t\u0103ng m\u1ea1nh!!!!!!\n\n"
    "Li\u00ean h\u1ec7: toasoan@baovietnam.vn ho\u1eb7c www.baovietnam.vn</p>"
)

result = clean_text(sample)
print("INPUT :")
print(sample)
print("\nOUTPUT:")
print(result)

INPUT :
<p><strong>Hà​Nội</strong>&nbsp;— Theo <a href='https://vnexpress.net'>nguồn tin</a>,
giá căn hộ năm 2024–2025 tăng mạnh!!!!!!

Liên hệ: toasoan@baovietnam.vn hoặc www.baovietnam.vn</p>

OUTPUT:
Hà Nội - Theo nguồn tin , giá căn hộ năm 2024-2025 tăng mạnh!! Liên hệ: hoặc


---

## Bước 5 — Lọc bài quá ngắn

### Cách làm
```python
MIN_CONTENT_LEN = 100   # ít nhất 100 ký tự
df = df[df["content"].str.len() >= MIN_CONTENT_LEN]
```

### Tại sao?
- Bài **< 100 ký tự** sau khi đã làm sạch thường là:
  - Bài lỗi, chỉ còn tiêu đề
  - Phần mô tả/thông báo ngắn ("Mời xem thêm tại...")
  - Bài crawl thiếu nội dung
- PhoBERT cần đủ ngữ cảnh để tạo ra embedding chất lượng. Bài quá ngắn cho embedding thiếu đặc trưng → **model bị nhiễu** khi học.

### Lợi ích
| | Số lượng | Ảnh hưởng |
|---|---|---|
| Giữ lại | Bài dài ≥ 100 ký tự | Embedding đủ thông tin |
| Loại bỏ | Bài quá ngắn/rỗng | Giảm nhiễu huấn luyện |

In [13]:
import numpy as np

MIN_CONTENT_LEN = 100

np.random.seed(42)
lengths = np.random.lognormal(mean=7.5, sigma=1.0, size=10000).astype(int)
lengths = np.clip(lengths, 5, 5000)

n_total = len(lengths)
n_too_short = (lengths < MIN_CONTENT_LEN).sum()
n_keep = (lengths >= MIN_CONTENT_LEN).sum()

print(f"T\u1ed5ng b\u00e0i           : {n_total:,}")
print(f"B\u00e0i < {MIN_CONTENT_LEN} k\u00fd t\u1ef1  : {n_too_short:,} ({n_too_short/n_total*100:.1f}%)")
print(f"B\u00e0i gi\u1eef l\u1ea1i        : {n_keep:,} ({n_keep/n_total*100:.1f}%)")
print(f"\n\u0110\u1ed9 d\u00e0i trung b\u00ecnh (to\u00e0n b\u1ed9): {lengths.mean():.0f} k\u00fd t\u1ef1")
print(f"\u0110\u1ed9 d\u00e0i trung b\u00ecnh (sau l\u1ecdc): {lengths[lengths >= MIN_CONTENT_LEN].mean():.0f} k\u00fd t\u1ef1")

Tổng bài           : 10,000
Bài < 100 ký tự  : 23 (0.2%)
Bài giữ lại        : 9,977 (99.8%)

Độ dài trung bình (toàn bộ): 2279 ký tự
Độ dài trung bình (sau lọc): 2284 ký tự


---

## Bước 6 — Loại bỏ bài trùng lặp (Deduplication)

### Cách làm
```python
# Tạo dedup key từ 100 ký tự đầu title + 200 ký tự đầu content
df["_dedup_key"] = df["title"].str[:100] + "|" + df["content"].str[:200]
df = df.drop_duplicates(subset=["_dedup_key"])
```

### Tại sao dùng title + content đầu thay vì so khớp toàn bộ?
- So khớp **toàn bộ** content: Bất kỳ sự khác biệt nhỏ nào (1 ký tự) sẽ không bị phát hiện là duplicate
- Chiến lược **prefix matching** (100+200 ký tự): Đủ để phân biệt hầu hết các bài, đồng thời vẫn bắt được các bài **re-syndicated** (bài đăng lại, chỉ khác phần cuối)
- Chạy **nhanh hơn** so với so khớp toàn bộ chuỗi dài

### Tại sao duplicate nguy hiểm?
1. **Data leakage**: Bài giống nhau xuất hiện cả train lẫn val → accuracy giả cao
2. **Overfit**: Model ghi nhớ bài cụ thể thay vì học pattern chung
3. **Thiên lệch**: Nhãn có nhiều bài viết lại sẽ bị overrepresented

In [14]:
import pandas as pd

sample_data = pd.DataFrame({
    "title": [
        "Gi\u00e1 x\u0103ng t\u0103ng 500 \u0111\u1ed3ng t\u1eeb ng\u00e0y mai",
        "Gi\u00e1 x\u0103ng t\u0103ng 500 \u0111\u1ed3ng t\u1eeb ng\u00e0y mai",
        "Gi\u00e1 x\u0103ng t\u0103ng 500 \u0111\u1ed3ng t\u1eeb ng\u00e0y 10/3",
        "GDP Vi\u1ec7t Nam qu\u00fd I t\u0103ng 6.5%",
    ],
    "content": [
        "Theo quy\u1ebft \u0111\u1ecbnh c\u1ee7a li\u00ean b\u1ed9, gi\u00e1 x\u0103ng E5 RON92 t\u0103ng 500 \u0111\u1ed3ng/l\u00edt...",
        "Theo quy\u1ebft \u0111\u1ecbnh c\u1ee7a li\u00ean b\u1ed9, gi\u00e1 x\u0103ng E5 RON92 t\u0103ng 500 \u0111\u1ed3ng/l\u00edt...",
        "Theo quy\u1ebft \u0111\u1ecbnh c\u1ee7a li\u00ean b\u1ed9, gi\u00e1 x\u0103ng E5 RON92 t\u0103ng 500 \u0111\u1ed3ng/l\u00edt, \u00e1p d\u1ee5ng t\u1eeb 15 gi\u1edd ng\u00e0y 10/3.",
        "T\u1ed5ng c\u1ee5c Th\u1ed1ng k\u00ea c\u00f4ng b\u1ed1 t\u1ed1c \u0111\u1ed9 t\u0103ng tr\u01b0\u1edfng GDP \u0111\u1ea1t 6.5% trong qu\u00fd \u0111\u1ea7u n\u0103m...",
    ],
    "label": ["kinh_doanh"] * 4,
})

sample_data["_dedup_key"] = sample_data["title"].str[:100] + "|" + sample_data["content"].str[:200]
deduped = sample_data.drop_duplicates(subset=["_dedup_key"]).drop(columns=["_dedup_key"])

print(f"Tr\u01b0\u1edbc dedup: {len(sample_data)} b\u00e0i")
print(f"Sau dedup  : {len(deduped)} b\u00e0i")
print(f"\u0110\u00e3 lo\u1ea1i    : {len(sample_data) - len(deduped)} b\u00e0i tr\u00f9ng")
print()
print(deduped[["title", "label"]].to_string(index=False))

Trước dedup: 4 bài
Sau dedup  : 3 bài
Đã loại    : 1 bài trùng

                              title      label
 Giá xăng tăng 500 đồng từ ngày mai kinh_doanh
Giá xăng tăng 500 đồng từ ngày 10/3 kinh_doanh
       GDP Việt Nam quý I tăng 6.5% kinh_doanh


---

## Bước 7 — Word Segmentation (Tách từ tiếng Việt)

### Cách làm
```python
from underthesea import word_tokenize

# Ghép title + description + content
combined = df["title"] + ". " + df["description"] + ". " + df["content"]

# Tách từ ghép tiếng Việt
df["text_segmented"] = combined.apply(
    lambda t: word_tokenize(t, format="text")
)
```

### Tại sao tách từ là quan trọng với tiếng Việt?

Tiếng Việt có **từ ghép** (multi-syllable words):
- `học sinh` = 1 từ (student) ≠ `học` (to learn) + `sinh` (born)
- `kinh tế` = 1 từ (economy) ≠ `kinh` (great) + `tế` (rescue)
- `ô tô` = 1 từ (car) ≠ `ô` (parasol) + `tô` (paint)

PhoBERT được huấn luyện trên corpus đã **tách từ** với `_` nối các âm tiết:
```
Không tách:  học sinh đến trường
Đã tách:     học_sinh đến trường
```

Nếu **không tách từ**, PhoBERT sẽ tokenize `học sinh` thành 2 token riêng biệt → mất đi ngữ nghĩa từ ghép → embedding kém chất lượng.

### Lợi ích
| Khía cạnh | Không tách từ | Có tách từ |
|---|---|---|
| Tokenization | `học`, `sinh` = 2 token | `học_sinh` = 1 token |
| Ngữ nghĩa | Mất nghĩa từ ghép | Bảo toàn nghĩa |
| Vocabulary coverage | Tăng OOV | Khớp pre-training vocab |
| Accuracy | Thấp hơn | Cao hơn ~2-5% |

> **Lưu ý thực tế:** Tách từ mất nhiều thời gian (vài giờ cho dataset lớn). Kết quả được lưu vào cột `text_segmented` để tái sử dụng.

In [15]:
examples = [
    "h\u1ecdc sinh \u0111\u1ebfn tr\u01b0\u1eddng h\u1ecdc ti\u1ec3u h\u1ecdc",
    "kinh t\u1ebf Vi\u1ec7t Nam t\u0103ng tr\u01b0\u1edfng m\u1ea1nh",
    "b\u1ed9 tr\u01b0\u1edfng b\u1ed9 t\u00e0i ch\u00ednh h\u1ecfp b\u00e1o",
    "x\u00e9t x\u1eed v\u1ee5 \u00e1n kinh t\u1ebf l\u1edbn nh\u1ea5t n\u0103m",
]

segmented_examples = [
    "h\u1ecdc_sinh \u0111\u1ebfn tr\u01b0\u1eddng_h\u1ecdc ti\u1ec3u_h\u1ecdc",
    "kinh_t\u1ebf Vi\u1ec7t_Nam t\u0103ng_tr\u01b0\u1edfng m\u1ea1nh",
    "b\u1ed9_tr\u01b0\u1edfng b\u1ed9 t\u00e0i_ch\u00ednh h\u1ecfp_b\u00e1o",
    "x\u00e9t_x\u1eed v\u1ee5_\u00e1n kinh_t\u1ebf l\u1edbn nh\u1ea5t n\u0103m",
]

print(f"{'Tr\u01b0\u1edbc t\u00e1ch t\u1eeb':<45} \u2192 {'Sau t\u00e1ch t\u1eeb'}")
print("-" * 90)
for before, after in zip(examples, segmented_examples):
    print(f"{before:<45} \u2192 {after}")

print()
print("Nh\u1eadn x\u00e9t: c\u00e1c t\u1eeb gh\u00e9p \u0111\u01b0\u1ee3c n\u1ed1i b\u1eb1ng '_' \u2192 PhoBERT nh\u1eadn d\u1ea1ng \u0111\u00fang l\u00e0 1 token")

Trước tách từ                                 → Sau tách từ
------------------------------------------------------------------------------------------
học sinh đến trường học tiểu học              → học_sinh đến trường_học tiểu_học
kinh tế Việt Nam tăng trưởng mạnh             → kinh_tế Việt_Nam tăng_trưởng mạnh
bộ trưởng bộ tài chính hỏp báo                → bộ_trưởng bộ tài_chính hỏp_báo
xét xử vụ án kinh tế lớn nhất năm             → xét_xử vụ_án kinh_tế lớn nhất năm

Nhận xét: các từ ghép được nối bằng '_' → PhoBERT nhận dạng đúng là 1 token


---

## Bước 8 — Xuất dữ liệu

### Schema đầu ra
```python
output_cols = ["id", "title", "description", "content", "text_segmented", "label", "url", "date"]
df_out.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
```

### Tại sao dùng `utf-8-sig` (UTF-8 with BOM)?
- `utf-8-sig` tự động thêm BOM signature `\xef\xbb\xbf` ở đầu file
- Excel và nhiều tool Windows nhận diện file là UTF-8 và hiển thị tiếng Việt đúng
- Pandas khi đọc lại với `encoding="utf-8-sig"` sẽ tự động bỏ qua BOM

### Cấu trúc dữ liệu cuối cùng

| Cột | Mô tả | Dùng cho |
|---|---|---|
| `id` | Index tuần tự | Tracking |
| `title` | Tiêu đề bài báo (đã làm sạch) | Input model |
| `description` | Mô tả ngắn / sapo | Input model |
| `content` | Nội dung bài báo (đã làm sạch) | Input model |
| `text_segmented` | title + desc + content đã tách từ | Input chính cho PhoBERT |
| `label` | Nhãn phân loại (10 nhãn) | Ground truth |
| `url` | URL gốc | Truy xuất nguồn |
| `date` | Ngày đăng | Metadata |

In [16]:
pipeline_steps = [
    ("1. \u0110\u1ecdc d\u1eef li\u1ec7u",          "Fallback encoding",             "Kh\u00f4ng crash d\u00f9 file d\u00f9ng encoding kh\u00e1c nhau"),
    ("2. Chu\u1ea9n ho\u00e1 c\u1ed9t",        "Rename + drop Unnamed",         "Code nh\u1ea5t qu\u00e1n, d\u1ec5 b\u1ea3o tr\u00ec"),
    ("3. \u00c1nh x\u1ea1 nh\u00e3n",          "30 category \u2192 10 nh\u00e3n",  "Nh\u00e3n r\u00f5 r\u00e0ng, gi\u1ea3m imbalance"),
    ("4. L\u00e0m s\u1ea1ch text",        "12 l\u1edbp x\u1eed l\u00fd",          "Lo\u1ea1i nhi\u1ec5u HTML/unicode, chu\u1ea9n ho\u00e1 k\u00fd t\u1ef1"),
    ("5. L\u1ecdc \u0111\u1ed9 d\u00e0i",           "Content >= 100 k\u00fd t\u1ef1",   "Lo\u1ea1i b\u00e0i r\u1ed7ng/thi\u1ebfu ng\u1eef c\u1ea3nh"),
    ("6. Deduplication",        "Title[:100] + Content[:200]",   "Ch\u1ed1ng data leakage, gi\u1ea3m overfit"),
    ("7. Word Segmentation",    "underthesea",                   "T\u1ed1i \u01b0u tokenization cho PhoBERT"),
    ("8. Xu\u1ea5t CSV",             "utf-8-sig encoding",            "T\u01b0\u01a1ng th\u00edch Windows/Excel"),
]

print(f"{'B\u01b0\u1edbc':<22} {'K\u1ef9 thu\u1eadt':<30} {'L\u1ee3i \u00edch ch\u00ednh'}")
print("=" * 95)
for step, technique, benefit in pipeline_steps:
    print(f"{step:<22} {technique:<30} {benefit}")

Bước                   Kỹ thuật                       Lợi ích chính
1. Đọc dữ liệu         Fallback encoding              Không crash dù file dùng encoding khác nhau
2. Chuẩn hoá cột       Rename + drop Unnamed          Code nhất quán, dễ bảo trì
3. Ánh xạ nhãn         30 category → 10 nhãn          Nhãn rõ ràng, giảm imbalance
4. Làm sạch text       12 lớp xử lý                   Loại nhiễu HTML/unicode, chuẩn hoá ký tự
5. Lọc độ dài          Content >= 100 ký tự           Loại bài rỗng/thiếu ngữ cảnh
6. Deduplication       Title[:100] + Content[:200]    Chống data leakage, giảm overfit
7. Word Segmentation   underthesea                    Tối ưu tokenization cho PhoBERT
8. Xuất CSV            utf-8-sig encoding             Tương thích Windows/Excel


---

## Tổng kết

### Tại sao tiền xử lý dữ liệu quan trọng đối với PhoBERT?

PhoBERT là mô hình **transfer learning** được pre-train trên corpus tiếng Việt sạch (Wikipedia, báo điện tử chất lượng cao). Khi fine-tune, dữ liệu đầu vào phải **càng gần với distribution pre-training corpus càng tốt**. Nếu dữ liệu còn nhiễu (HTML, Unicode sai, bài trùng lặp), mô hình sẽ:

1. **Lãng phí capacity** vào việc học pattern nhiễu thay vì nội dung thật
2. **Tạo embedding kém** do tokenizer gặp OOV tokens từ HTML/entities
3. **Generalize kém** do overfit trên dữ liệu lặp
4. **Đánh giá sai** do data leakage giữa train/val/test

### Kết quả sau pipeline

```
Input  : ~X bài thô, nhiều noise, 30+ category
Output : Dữ liệu sạch, 10 nhãn rõ ràng
         ✓ Unicode chuẩn NFC
         ✓ Không HTML, URL, email
         ✓ Từ đã tách (word_segmented)
         ✓ Không duplicate
         ✓ Content có độ dài tối thiểu
         → Sẵn sàng cho train_v4.py
```

### Quy tắc "Garbage In, Garbage Out"

> Mô hình học máy chỉ tốt như dữ liệu nó được huấn luyện. Đầu tư vào tiền xử lý dữ liệu thường cho ROI cao hơn so với tối ưu hoá kiến trúc mô hình.

---
*Báo cáo này mô tả `prepare_articles.py` — bước chuẩn bị dữ liệu cho pipeline huấn luyện PhoBERT phân loại tin tức tiếng Việt.*